# Publish QuickTalk run 8 on Hugging Face as **LMLM_97M_1**

Runtime: **CPU is enough** (no GPU needed, lowest cost). It takes about 3-5 minutes, mostly the upload (~800 MB).

What it uploads: the chat model and the base model as safetensors, `config.json`, `tokenizer.json`, `quicktalk_lm.py`,
`inference.py` and the model card `README.md`. **No training data is uploaded.**

**Token, before you run:** click the key icon (Secrets) in Colab's left bar, add a secret named `HF_TOKEN` with your
Hugging Face *write* token, and switch on "Notebook access". If there is no secret, cell 3 asks for the token in a
hidden box. Never paste the token into a cell: this notebook is in a public repo.

In [ ]:
# 1. Drive, code, libraries
import os, subprocess, shutil, json
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/quicktalk_run8'; SRC = '/content/qt_pub'; OUT = '/content/LMLM_97M_1'   # local disk
REPO_NAME = 'LMLM_97M_1'
PRIVATE = False          # True = only you can see the model on Hugging Face
for f in ('checkpoints/sft_final.pt', 'checkpoints/pretrain_final.pt', 'tokenizer.json'):
    assert os.path.exists(f'{WORK}/{f}'), f'missing {WORK}/{f}'
if os.path.exists(SRC): subprocess.run(['git', '-C', SRC, 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'claude/intelligent-ride-oxqjfy', '--filter=blob:none',
                    '--sparse', 'https://github.com/sraivante/quicktalk.git', SRC], check=True)
    subprocess.run(['git', '-C', SRC, 'sparse-checkout', 'set', 'train/colab', 'hf/LMLM_97M_1'], check=True)
!pip -q install -U huggingface_hub safetensors tokenizers

In [ ]:
# 2. Build the upload folder on the local disk: weights -> safetensors (weights only, no optimizer state)
import torch
from safetensors.torch import save_file
os.makedirs(OUT, exist_ok=True)
for ckpt, name in (('sft_final.pt', 'model.safetensors'), ('pretrain_final.pt', 'base_model.safetensors')):
    s = torch.load(f'{WORK}/checkpoints/{ckpt}', map_location='cpu', weights_only=False)
    save_file({k: v.float().contiguous() for k, v in s['model'].items()}, f'{OUT}/{name}', metadata={'format': 'pt'})
    cfg = s['cfg']; print(ckpt, '->', name, 'stage', s.get('stage'), 'step', s.get('step')); del s
json.dump(cfg, open(f'{OUT}/config.json', 'w'), indent=1)
shutil.copy(f'{WORK}/tokenizer.json', OUT)
shutil.copy(f'{SRC}/train/colab/quicktalk_lm.py', OUT)
for f in ('README.md', 'inference.py'): shutil.copy(f'{SRC}/hf/LMLM_97M_1/{f}', OUT)
# smoke test: the uploaded files load and answer
print(subprocess.run(['python', f'{OUT}/inference.py', 'Hello!'], capture_output=True, text=True).stdout)
print(subprocess.run(['python', f'{OUT}/inference.py', 'Please correct this: My sister play badminton every evening.'],
                     capture_output=True, text=True).stdout)
!ls -la {OUT}

In [ ]:
# 3. Log in (token from the Colab secret HF_TOKEN, else a hidden prompt; it is never printed or saved)
from huggingface_hub import HfApi
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None
if not token:
    from getpass import getpass
    token = getpass('Hugging Face write token: ')
api = HfApi(token=token)
user = api.whoami()['name']; repo_id = f'{user}/{REPO_NAME}'
print('logged in as', user, '-> will publish', repo_id, '(private)' if PRIVATE else '(public)')

In [ ]:
# 4. Upload (the model card's example gets your real repo id)
readme = f'{OUT}/README.md'
card = open(readme).read().replace('<your-username>/LMLM_97M_1', repo_id)   # read first, then write
open(readme, 'w').write(card)
shutil.rmtree(f'{OUT}/__pycache__', ignore_errors=True)
api.create_repo(repo_id, repo_type='model', private=PRIVATE, exist_ok=True)
api.upload_folder(repo_id=repo_id, folder_path=OUT, commit_message='LMLM_97M_1: QuickTalk run 8 chat + base model, model card')
print('done: https://huggingface.co/' + repo_id)

In [ ]:
# 5. Check: download from Hugging Face into a fresh folder and chat once
from huggingface_hub import snapshot_download
p = snapshot_download(repo_id, token=token, local_dir='/content/hf_check')
print(subprocess.run(['python', f'{p}/inference.py', 'Good morning! How are you today?'], capture_output=True, text=True).stdout)